# 035 — Programación probabilística y causalidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia en 5 minutos

**Programación probabilística:** el modelo generativo se escribe como un **programa** con instrucciones de muestreo (`x ~ Normal(0,1)`) y de condicionamiento (`observe(y == 3)`); un motor de inferencia genérico calcula el posterior sobre las variables latentes. Ventaja: separa *modelado* (el programa) de *inferencia* (rechazo, importancia, MCMC — clase 031).

**La escalera de la causalidad (Pearl):**

```text
Peldaño 1 — VER      : P(Y | X)        asociación, lo que da cualquier dataset
Peldaño 2 — HACER    : P(Y | do(X=x))  intervención: fijar X cortando sus causas
Peldaño 3 — IMAGINAR : contrafactuales ¿qué habría pasado si...?
```

`P(Y | X=x)` ≠ `P(Y | do(X=x))` cuando existe un **confusor** Z que causa a ambos: condicionar *selecciona* subpoblaciones; intervenir *rompe* la flecha Z→X.

**Ajuste por puerta trasera (backdoor):** si Z bloquea todos los caminos espurios,

```text
P(Y | do(X=x)) = Σ_z P(Y | X=x, Z=z) · P(z)
```

— se promedia el efecto por estrato con los pesos *poblacionales* de Z, no los condicionales. La paradoja de Simpson (la asociación se invierte al estratificar) se resuelve exactamente así.


### Mini ejemplo

Un programa generativo de 3 líneas: `z ~ Bernoulli(0.5); x ~ Bernoulli(0.8 si z else 0.2); y ~ Bernoulli(0.7 si z else 0.3)`. Aquí X e Y están correlacionados sin que X cause a Y (confusor Z). `P(Y|X=1)` es alto, pero `P(Y|do(X=1)) = P(Y) = 0.5`: la intervención revela que la palanca X no sirve. El laboratorio `probability` muestrea modelos de este estilo con semilla fija.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("probability", seed=35)
show(result)


## Reflexión

1. En el modelo del laboratorio, ¿qué consulta es asociacional y cuál requeriría una intervención real o un supuesto causal explícito para responderse?
2. ¿Por qué ningún volumen de datos observacionales, por sí solo, permite subir del peldaño 1 al 2 de la escalera? ¿Qué información extra hace falta?
3. Da un ejemplo donde condicionar por una variable (un colisionador, clase 027) *cree* una correlación espuria en lugar de eliminarla. ¿Qué implica para "controlar por todo"?
